# Final Insights Synthesis

## Step 1: Load Data and Previous Results

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../datasets/student_exam_clustered.csv')
print(f"Dataset: {df.shape}, Clusters: {sorted(df['cluster'].unique())}")

Dataset: (10000, 36), Clusters: [0, 1, 2]


## Step 2: Top Predictors Summary

In [2]:
import json
from scipy import stats

# Load model results produced by notebook 07 (single source of truth)
with open('../datasets/model_results.json') as f:
    model_results = json.load(f)

print("=" * 60)
print("TOP PREDICTORS OF STUDENT PERFORMANCE")
print("=" * 60)

notes = {
    'previous_gpa': 'Metrik historis, tidak dapat diintervensi',
    'study_attendance': 'Fitur interaksi perilaku (study x attendance)',
    'study_hours_per_day': 'Prediktor perilaku terkuat',
    'social_study_ratio': 'Rasio distraksi vs usaha belajar',
    'productive_hours': 'Total waktu produktif harian',
    'engagement_score': 'Rata-rata keterlibatan perilaku',
    'social_media_hours': 'Distraksi media sosial',
}

print(f"\n{'Feature':<28} {'RF Importance':<15} {'Pearson r':<12} {'Note'}")
print('-' * 80)
for feat, imp in model_results['feature_importance'].items():
    if feat in df.columns:
        r = stats.pearsonr(df[feat], df['final_exam_score'])[0]
        r_str = f'{r:.4f}'
    else:
        r_str = 'N/A'
    print(f"{feat:<28} {imp:<15.4f} {r_str:<12} {notes.get(feat, '')}")

TOP PREDICTORS OF STUDENT PERFORMANCE

Feature                      RF Importance   Pearson r    Note
--------------------------------------------------------------------------------
previous_gpa                 0.4212          0.8912       Metrik historis, tidak dapat diintervensi
study_attendance             0.0969          0.5914       Fitur interaksi perilaku (study x attendance)
social_study_ratio           0.0790          -0.4491      Rasio distraksi vs usaha belajar
study_hours_per_day          0.0788          0.5758       Prediktor perilaku terkuat
productive_hours             0.0555          0.4563       Total waktu produktif harian


## Step 3: Hypothesis Results Summary

In [3]:
print("=" * 60)
print("HYPOTHESIS RESULTS SUMMARY")
print("=" * 60)

# --- Compute H3 from clustered data (practical-significance rule) ---
cluster_scores = df.groupby('cluster')['final_exam_score'].mean()
cluster_pass = df.groupby('cluster')['pass_fail_enc'].mean()
score_range = cluster_scores.max() - cluster_scores.min()
pass_range = cluster_pass.max() - cluster_pass.min()
h3_result = "DITERIMA" if score_range > 5 or pass_range > 0.10 else "DITOLAK"

# --- H2 result loaded from notebook 07 artifact (no hardcoded values) ---
h2 = model_results['h2']

hypotheses = {
    'H1': {
        'definition': 'study_hours (+) & social_media (-) berkorelasi dengan final_exam_score',
        'result': 'DITERIMA',
        'evidence': 'r_study=+0.5758 (p<0.001), r_social=-0.2463 (p<0.001)'
    },
    'H2': {
        'definition': 'Random Forest > Logistic Regression dalam prediksi kelulusan',
        'result': h2['result'],
        'evidence': (f"corrected resampled t-test: t={h2['t_corr']:.3f}, p={h2['p_corr']:.3f} "
                     f"(df={h2['df']}); LR CV={model_results['lr']['cv']:.4f} vs "
                     f"RF CV={model_results['rf']['cv']:.4f}")
    },
    'H3': {
        'definition': 'Siswa dapat dikelompokkan ke cluster berdasarkan pola perilaku',
        'result': h3_result,
        'evidence': (f"score range={score_range:.1f} (<5), pass-rate range={pass_range:.1%} (<10%); "
                     f"signifikan statistik namun effect size negligible -> tidak bermakna praktis")
    }
}

for h, info in hypotheses.items():
    print(f"\n{h}: {info['definition']}")
    print(f"  Status: {info['result']}")
    print(f"  Evidence: {info['evidence']}")

print(f"\n--- H3 Cluster Detail ---")
for c in sorted(df['cluster'].unique()):
    n = df[df['cluster'] == c].shape[0]
    print(f"  Cluster {c} (n={n}): score={cluster_scores[c]:.1f}, pass_rate={cluster_pass[c]:.1%}")
print(f"\nH3 Final: {h3_result}")

HYPOTHESIS RESULTS SUMMARY

H1: study_hours (+) & social_media (-) berkorelasi dengan final_exam_score
  Status: DITERIMA
  Evidence: r_study=+0.5758 (p<0.001), r_social=-0.2463 (p<0.001)

H2: Random Forest > Logistic Regression dalam prediksi kelulusan
  Status: TIDAK DIDUKUNG (LR > RF, signifikan)
  Evidence: corrected resampled t-test: t=-2.996, p=0.040 (df=4); LR CV=0.8610 vs RF CV=0.8562

H3: Siswa dapat dikelompokkan ke cluster berdasarkan pola perilaku
  Status: DITOLAK
  Evidence: score range=2.4 (<5), pass-rate range=7.2% (<10%); signifikan statistik namun effect size negligible -> tidak bermakna praktis

--- H3 Cluster Detail ---
  Cluster 0 (n=4008): score=50.7, pass_rate=51.2%
  Cluster 1 (n=3274): score=48.3, pass_rate=44.0%
  Cluster 2 (n=2718): score=49.7, pass_rate=50.2%

H3 Final: DITOLAK


## Step 4: Model Performance Summary

In [4]:
print("=" * 60)
print("CLASSIFICATION MODEL PERFORMANCE (H2)")
print("=" * 60)

# Loaded from notebook 07 artifact — tuned models (no hardcoded values)
models = {
    'Logistic Regression': model_results['lr'],
    'Random Forest': model_results['rf'],
}

print(f"\n{'Model':<22} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'CV Acc':<12}")
print('-' * 82)
for name, m in models.items():
    print(f"{name:<22} {m['accuracy']:<12.4f} {m['precision']:<12.4f} {m['recall']:<12.4f} {m['f1']:<12.4f} {m['cv']:<12.4f}")

print(f"\nROC-AUC: " + ", ".join(f"{k}={v:.4f}" for k, v in model_results['roc_auc'].items()))
print(f"\nH2 ({model_results['h2']['result']}): corrected resampled t-test "
      f"p={model_results['h2']['p_corr']:.3f} (df={model_results['h2']['df']}); "
      f"hipotesis directional 'RF > LR' tidak didukung.")

CLASSIFICATION MODEL PERFORMANCE (H2)

Model                  Accuracy     Precision    Recall       F1           CV Acc      
----------------------------------------------------------------------------------
Logistic Regression    0.8705       0.8780       0.8519       0.8648       0.8610      
Random Forest          0.8605       0.8651       0.8447       0.8548       0.8562      

ROC-AUC: LR (Default)=0.9456, RF (Default)=0.9370, LR (Tuned)=0.9457, RF (Tuned)=0.9376

H2 (TIDAK DIDUKUNG (LR > RF, signifikan)): corrected resampled t-test p=0.040 (df=4); hipotesis directional 'RF > LR' tidak didukung.


## Step 5: Cluster Profiles Summary

In [5]:
print("=" * 60)
print("CLUSTER PROFILES (H3)")
print("=" * 60)

cluster_features = ['study_hours_per_day', 'attendance_rate', 'sleep_hours',
                    'social_media_hours', 'assignment_completion_rate',
                    'online_courses_completed']
profile_cols = cluster_features + ['previous_gpa', 'final_exam_score', 'pass_fail_enc']

cluster_profiles = df.groupby('cluster')[profile_cols].mean().round(2)
print(cluster_profiles.to_string())

print("\nCluster Interpretation:")
for c in sorted(df['cluster'].unique()):
    n = df[df['cluster']==c].shape[0]
    score = cluster_profiles.loc[c, 'final_exam_score']
    study = cluster_profiles.loc[c, 'study_hours_per_day']
    social = cluster_profiles.loc[c, 'social_media_hours']
    pass_rate = cluster_profiles.loc[c, 'pass_fail_enc']
    
    if study > cluster_profiles['study_hours_per_day'].mean() and score > cluster_profiles['final_exam_score'].mean():
        label = "High Performers"
    elif study < cluster_profiles['study_hours_per_day'].mean() and score < cluster_profiles['final_exam_score'].mean():
        label = "At-Risk Students"
    else:
        label = "Moderate Students"
    
    print(f"\n  Cluster {c} (n={n}): {label}")
    print(f"    Score: {score:.1f}, Study hrs: {study:.1f}, Social media: {social:.1f}, Pass rate: {pass_rate:.1%}")

CLUSTER PROFILES (H3)
         study_hours_per_day  attendance_rate  sleep_hours  social_media_hours  assignment_completion_rate  online_courses_completed  previous_gpa  final_exam_score  pass_fail_enc
cluster                                                                                                                                                                            
0                       2.78            85.02         6.85                1.96                       88.08                      1.32          2.02             50.75           0.51
1                       3.31            84.24         7.21                3.25                       68.30                      1.39          1.94             48.33           0.44
2                       3.03            84.81         7.03                2.45                       80.30                      3.85          1.99             49.74           0.50

Cluster Interpretation:

  Cluster 0 (n=4008): Moderate Students
    Score: 5

## Step 6: Key Insights and Overall Recommendations

In [6]:
print("=" * 60)
print("KEY INSIGHTS")
print("=" * 60)

_acc = (model_results['lr']['accuracy'] + model_results['rf']['accuracy']) / 2
insights = [
    "1. study_hours_per_day adalah prediktor perilaku terkuat (r=0.58) — H1 validated",
    "2. social_media_hours berkorelasi negatif signifikan (r=-0.25) — menghambat performa",
    "3. previous_gpa mendominasi prediksi (r=0.89) tapi tidak dapat diintervensi langsung",
    "4. Faktor demografis (parental education, tutoring, family income) TIDAK signifikan",
    f"5. LR & RF setara (~{_acc:.1%} accuracy); hipotesis 'RF > LR' (H2) TIDAK didukung — "
    "interpretabilitas LR menjadi keunggulan praktis",
    "6. Clustering perilaku TIDAK menghasilkan segmen terdiferensiasi (H3 ditolak; silhouette ~0.11, "
    "eta^2 negligible) — intervensi berbasis aturan dari H1 lebih tepat daripada segmentasi cluster",
]

for insight in insights:
    print(f"  {insight}")

print(f"\n{'=' * 60}")
print("OVERALL RECOMMENDATIONS")
print("=" * 60)

recommendations = [
    "1. Implementasikan program belajar terstruktur — study_hours adalah prediktor perilaku terkuat (H1 validated)",
    "2. Lakukan kampanye kesadaran penggunaan media sosial — korelasi negatif signifikan terkonfirmasi (H1 validated)",
    "3. Fokus intervensi pada faktor perilaku individual (bukan segmentasi cluster) — faktor demografis tidak signifikan, cluster tidak terdiferensiasi",
    "4. Pilih LR sebagai model utama untuk interpretabilitas — setara dengan RF secara performa",
]

for rec in recommendations:
    print(f"  {rec}")

KEY INSIGHTS
  1. study_hours_per_day adalah prediktor perilaku terkuat (r=0.58) — H1 validated
  2. social_media_hours berkorelasi negatif signifikan (r=-0.25) — menghambat performa
  3. previous_gpa mendominasi prediksi (r=0.89) tapi tidak dapat diintervensi langsung
  4. Faktor demografis (parental education, tutoring, family income) TIDAK signifikan
  5. LR & RF setara (~86.6% accuracy); hipotesis 'RF > LR' (H2) TIDAK didukung — interpretabilitas LR menjadi keunggulan praktis
  6. Clustering perilaku TIDAK menghasilkan segmen terdiferensiasi (H3 ditolak; silhouette ~0.11, eta^2 negligible) — intervensi berbasis aturan dari H1 lebih tepat daripada segmentasi cluster

OVERALL RECOMMENDATIONS
  1. Implementasikan program belajar terstruktur — study_hours adalah prediktor perilaku terkuat (H1 validated)
  2. Lakukan kampanye kesadaran penggunaan media sosial — korelasi negatif signifikan terkonfirmasi (H1 validated)
  3. Fokus intervensi pada faktor perilaku individual (bukan segmentas